# POC Extraction de données

Extraction des données provenant des rapport de G2K vers un format plus simple.

Import, définition des variables et récupération du contenue du rapport (sous forme de `string`)

In [ ]:
import pandas as pd
import re

## Variable global
content = ""
sections_content = {}
data_report = {}
sections_titles = []

# Récuperaction du contenue du rapport
with open("../data/rapport-RGU_3cm.txt", "r") as f:
    content = f.read()

Définition de mes *regex*

In [ ]:
section_header_pattern        = re.compile(r"^\*+$\n^\*{5}(.*)\*{5}$\n^\*+$", re.MULTILINE)
s1_kv_pattern                 = re.compile(r"^([A-ZÀ-Ÿ][a-zA-ZÀ-ÿ' ]+?)\s*:\s*(.*)$", re.MULTILINE)
s2_header_pattern             = re.compile(r"(Numéro)\s+(Début)\s+-\s+(Fin)\s+(Centroïde)\s+(Energie)\s+(FWHM)\s+(Surface)\s+(Incert\.)\s+(Fond sous)\s*\r?\n^\s*(du pic)\s+(\(canaux\))\s+(\(keV\))\s+(\(keV\))\s+(le pic)", re.MULTILINE)
s2_data_pattern               = re.compile(r"^\s*([MmF]?)\s*(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+-\s*(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)\s+(\d+(?:\.\d+)?(?:E[+\-]?\d+)?)", re.MULTILINE)
s3_header_pattern             = re.compile(r"^\s*(Nom du.*)$\n^(.*)$", re.MULTILINE)
s3_word_column_pattern        = re.compile(r"([A-Za-zÀ-ÿ]+\.?)")
s6_header_pattern             = re.compile(r"^\s+(.*)$\n^\s+(.*)$", re.MULTILINE)
s6_word_column_pattern        = re.compile(r"([A-Za-zÀ-ÿ]+\.?)")
s6_nucleide_line_pattern      = re.compile(r"^[+>?]\s+([A-Z]{1,2}-\d{1,3})\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)\s+(\S+)", re.MULTILINE)


## Extraction des header de chaque section
1 - On extrait les titre et on créer un dictionnaire de DataFrame

2- On découpe `content` par section que l'on range dans un dictionnaire `sections_content`

In [ ]:
section_headers = section_header_pattern.findall(content)

# 1
for i, section in enumerate(section_headers):
    sections_titles.append(section.strip())
    data_report[section.strip()] = None

# 2
matches = list(section_header_pattern.finditer(content))
for i, match in enumerate(matches):
    title = match.group(1).strip()
    start = match.end()
    end = matches[i + 1].start() if i + 1 < len(matches) else len(content)
    sections_content[title] = content[start:end].strip()

### S1 - RAPPORT DE L’ANALYSE DU SPECTRE
C'est les metadata du rapport.

Pour l'instant ce n'est pas encore bon:
Mauvaise dections des clés/valeurs

In [ ]:
content_s1 = sections_content[sections_titles[0]]
matches = s1_kv_pattern.findall(content_s1)
data_report[sections_titles[0]] = {key.strip(): value.strip() for key, value in matches}

####### DATA ##########
data_report[sections_titles[0]]

### S2 - RAPPORT ANALYSE DES PICS

In [49]:
def extract_header_s2(content):
    match = re.search(s2_header_pattern, content)
  
    if not match:
        return None
        
    columns = [
        "Marker",                                   # Marqueur (M, m, F)
        f"{match.group(1)} {match.group(9)}",       # Numéro du pic
        f"{match.group(2)} {match.group(10)}",        # Début (canaux)       
        f"{match.group(3)} {match.group(11)}",                       # Fin (canaux)
        match.group(4),                             # Centroïde
        f"{match.group(5)} {match.group(12)}",      # Energie (keV)
        f"{match.group(6)} {match.group(13)}",      # FWHM (keV)
        match.group(7),                             # Surface
        match.group(8),                             # Incert.
        f"{match.group(8)} {match.group(13)}"       # Fond sous le pic
    ]
    
    return columns



In [51]:
def extract_data_s2(content, header):
    matches = re.findall(s2_data_pattern, content)
    data = pd.DataFrame(matches, columns=header)
    data[header[1:]] = data[header[1:]].apply(pd.to_numeric, errors="coerce")
    
    return data


In [ ]:
content_s2 = sections_content[sections_titles[1]]
header_s2 = extract_header_s2(content_s2)
data_report[sections_titles[1]] = extract_data_s2(content_s2, header_s2)

####### DATA ##########
data_report[sections_titles[1]]

[('', '1', '80', '97', '89.94', '16.21', '1.27', '9.406E+03', '650.88', '2.958E+04'), ('', '2', '98', '114', '106.38', '19.26', '1.43', '4.954E+03', '612.65', '2.841E+04'), ('M', '3', '130', '158', '141.01', '25.67', '0.85', '4.291E+03', '269.06', '2.308E+04'), ('m', '4', '130', '158', '150.25', '27.38', '0.86', '3.786E+03', '266.29', '2.492E+04'), ('', '5', '191', '205', '200.49', '36.68', '0.50', '5.911E+02', '525.79', '2.378E+04'), ('', '6', '242', '262', '253.74', '46.53', '0.88', '8.341E+04', '922.62', '3.544E+04'), ('M', '7', '264', '298', '272.26', '49.96', '0.92', '3.473E+03', '266.89', '2.567E+04'), ('m', '8', '264', '298', '289.91', '53.23', '0.93', '9.527E+03', '333.34', '2.869E+04'), ('M', '9', '306', '353', '325.38', '59.79', '0.91', '3.449E+03', '279.02', '2.988E+04'), ('m', '10', '306', '353', '332.87', '61.18', '0.91', '3.955E+03', '283.48', '3.258E+04'), ('m', '11', '306', '353', '344.21', '63.28', '0.92', '9.097E+04', '671.92', '3.559E+04'), ('M', '12', '356', '447', 

### S3 - RAPPORT IDENTIFICATION DES NUCLEIDES

In [ ]:
def extract_header_s3(content):
    header = []

    matches = re.findall(s3_header_pattern, content)
    lines = matches[0]
    
    print(lines)

    l1 = re.findall(s3_word_column_pattern, lines[0])
    l2 = re.findall(s3_word_column_pattern, lines[1])
    
    print(l1)
    print(l2)
    
    # Fusion en partant de la fin
    for a, b in zip(reversed(l1), reversed(l2)):
        header.insert(0, f"{a} {b}".strip())

    # Si l1 est plus longue, on conserve le début restant
    if len(l1) > len(l2):
        header = l1[: len(l1) - len(l2)] + header

    return header

In [ ]:
def extract_data_s3(content):
    pass

In [ ]:
content_s3 = sections_content[sections_titles[2]]
header = extract_header_s3(content_s3)

### S4 - RAPPORT IDENTIFICATION AVEC CORRECTION D’INTERFERENCE

In [ ]:
content_s4 = sections_content[sections_titles[3]]

### S5 - RAPPORT LIMITES DE DETECTION

In [ ]:
content_s5 = sections_content[sections_titles[4]]

### S6 - RAPPORT LIMITES DE DETECTION  ISO 11929

In [ ]:
def extract_header_s6(sections_content):
    header = []
    str = sections_content[sections_titles[5]]

    lignes = s6_header_pattern.findall(str)

    l1 = re.findall(s6_word_column_pattern, lignes[0][0])
    l2 = re.findall(s6_word_column_pattern, lignes[0][1])
    
    print(l1)
    print(l2)
    # Fusion en partant de la fin
    for a, b in zip(reversed(l1), reversed(l2)):
        header.insert(0, f"{a} {b}".strip())

    # Si l1 est plus longue, on conserve le début restant
    if len(l1) > len(l2):
        header = l1[: len(l1) - len(l2)] + header

    return header


header = extract_header_s6(sections_content)

lignes = re.findall(s6_nucleide_line_pattern, sections_content[sections_titles[5]])

data_report["RAPPORT LIMITES DE DETECTION  ISO 11929"] = pd.DataFrame(
    lignes, columns=header
)
df = data_report["RAPPORT LIMITES DE DETECTION  ISO 11929"]

df[header[1:]] = df[header[1:]].apply(pd.to_numeric, errors="coerce")

# Zone de Test

In [ ]:
import seaborn as sns

sns.pairplot(data_report[sections_titles[1]])